# Goal

Серия экспериментов. Треним за 5 поколений по 6 млн шагов с последующим отбором чемпионов. Наследники этих чемпионов будут использоваться на следующем шаге.

Данный эксперимент - это ВТОРОЙ шаг, альтернативная цепочка параллельно `17e_study_20.2` и `17e_study_20.2b`.

Гипотеза из `17e_study_20.2b` в том, что агенту не хватает горизонта планирования, когда он слева от иглу. И поэтому он предпочитает спрыгивать на льдину вниз, вместо того, чтобы бежать направо. А что если агента поставить поближе к иглу в этом сценарии?

# set_hyperparameters

In [3]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    generations_count = 6
    generation_ind = 1
    generation_steps_count = 6_000_000
    all_generations_steps_count = generation_steps_count * generations_count
    learn_rate_range = (0.00025, 0.00025 * 0.1)
    ent_coef_range = (0.05, 0.05 * 0.1)
    tau_range = (0.5, 0.1)
    ####
    
    import random
    HP.system.random_seed = random.randint(1, 100)
    HP.system.is_torch_deterministic = True
    HP.system.is_torch_compile = True
    
    HP.env.ident = 'FrostbiteNoFrameskip-v4'
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6
    HP.env.idle_penalty = 0
    HP.env.life_lost_penalty = 0

    HP.agent.parent = optuna_trial.suggest_categorical('agent.parent', [
        '17e_ppo_tr_atari_mp_14:118',
        '17e_ppo_tr_atari_mp_14:113',
        '17e_ppo_tr_atari_mp_13:335',
    ])
    HP.agent.layers_count = 3 # number of transformer layers
    HP.agent.heads_count = 4 # number of heads used in multi-head attention
    HP.agent.d_model = 256 # dimension of the transformer
    HP.agent.obs_sequence_length = 4 # length observation chain agent incepts
    HP.agent.action_plan_length = 10 # number of actions agent must think upfront about
    HP.agent.positional_encoding = 'learned' # positional encoding type of the transformer: "", "absolute", "learned"
    
    # Video params
    HP.video.capture_policy = 'every(500000)' # video capture policy depending on steps
    HP.video.capture_env_states = None
    HP.video.capture_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level5_101:1', # bear level, night
    ]
    HP.video.capture_env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'],
    ]
    HP.video.break_on_level_passed = True
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = generation_steps_count # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_states = None
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level5_101:1', # bear level, night
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'],
    ]
    
    tau_change_speed = (tau_range[1] - tau_range[0]) / generations_count
    tau_a = tau_range[0] + tau_change_speed * generation_ind
    tau_b = tau_a + tau_change_speed
    HP.ppo.tau = f'linear({tau_a}, {tau_b})' # temperature to inject randomness during actions selection (Gumbel Max)
    # HP.ppo.tau = 'linear(0.5, 0.34)' # temperature to inject randomness during actions selection (Gumbel Max)
    
    HP.ppo.epochs_count = 2 
    HP.ppo.minibatches_count = 8
    learn_rate_change_speed = (learn_rate_range[1] - learn_rate_range[0]) / generations_count
    learn_rate_a = learn_rate_range[0] + learn_rate_change_speed * generation_ind
    learn_rate_b = learn_rate_a + learn_rate_change_speed
    HP.ppo.learn_rate = f'linear({learn_rate_a}, {learn_rate_b})'
    # HP.ppo.learn_rate = 'linear(0.00025, 0.00015)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.5 # coefficient of the value function within loss function
    ent_coef_change_speed = (ent_coef_range[1] - ent_coef_range[0]) / generations_count
    ent_coef_a = ent_coef_range[0] + ent_coef_change_speed * generation_ind
    ent_coef_b = ent_coef_a + ent_coef_change_speed
    HP.ppo.ent_coef = f'linear({ent_coef_a}, {ent_coef_b})' # coefficient of the entropy member within loss function
    # HP.ppo.ent_coef = 'linear(0.05, 0.03)' # coefficient of the entropy member within loss function
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True # Toggles whether or not to use a clipped loss for the value function, as per the paper
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # e target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP
# @launchit.stop

In [4]:
from unittest.mock import Mock
HP = Mock()
set_hyperparameters(HP, None, None)
HP.ppo.tau, HP.ppo.learn_rate, HP.ppo.ent_coef

('linear(0.43333333333333335, 0.3666666666666667)',
 'linear(0.00021250000000000002, 0.00017500000000000003)',
 'linear(0.0425, 0.035)')

# Results
<TBD>

С одной стороны стало лучше, по сравнению с `17e_study_20.2b` - агент теперь научился проходить 3 вот этих сценария:
- `[..., 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo']`
- `[..., 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo']`
- `[..., 'bear_chases_bailey_to_the_right_of_igloo']`

Т.е. агент, будучи справа или слева, может заходить в мигающий иглу. Но вот в этот сценарий, в котором надо пробежать чуть более длинную дистанцию, агент так и не научился `[..., 'bear_chases_bailey_to_the_left_of_igloo']`. Получается, что, действительно, пипетка коротковата (горизонт планирования). Пришла в голову такая мысль. А ведь 10 шагов - это не залог того, что агент построит именно красивую, правильную траекторию. Вероятность, построить например, Г-образную траекторию (вперед-вперед-вперед-...-наверх) крайне низка: 1 / 6^10. Поэтому короткие тракетории ещё имеют шанс выработаться и закрепиться, а вот чуть-чуть длинные - не получается. И даже регуляризация на совпадение тракеторий не помогает. Последние шаги тракеторий (5,6 и т.д.) скорее всего гуляют и виляют, как хвост у собаки. О, кстати, хорошая аналогия. В основании хвоста ещё есть какая-то предсказуемость, а на кончике хвоста начинается вакханалия. М.б. это и есть причина близорукости?

Да и раньше я подмечал, что агент, когда иглу построена, а он сам находится слева от иглу, то он часто это делает через льдины, т.е. сначла спрыгивает на них, а потом по диагонали поднимается.

<img src="./img/levels_passed.png">
<img src="./img/reward.png">
<img src="./img/episode_r.png">

**Выводы**
1) фактор "близорукости" (малый горизонт планирования) присутствует. Ключевой он или нет - хз, но вот видно, что если требуется спланировать чуть дальше, чем несколько шагов, то у агента возникают проблемы
2) подумать над другими механизмами планирования